## **RDD Transformations and Actions**

### **Spark 설정하기**

In [ ]:
"""
    [+] SparkConf 객체 생성 및 설정
    - setMaster(), 마스터 노드 위치: "local"
    - 앱이름: "transformations_actions"
"""

conf = SparkConf()\
    .setMaster('local')\
    .setAppName('rdd-transformations-and-actions')

In [ ]:
# [+] SparkContext 객체 생성
sc = SparkContext(conf=conf)

In [ ]:
# SpackContext 객체 하나 더 생성하기(과연?)
sc = SparkContext(conf=conf)

In [ ]:
# SparkContext는 프로그램 내에서 하나의 객체만 생성할 수 있다. (= Singleton 패턴)
# SparkContext 객체 해제하기
sc.stop()

In [ ]:
# 다시 생성하기
sc = SparkContext(conf=conf)

### **기본적인 액션 메서드들**
+ ```collect()```: 모든 파티션의 데이터들을 드라이버 노드로 수집 및 리스트로 반환
+ ```countByValue()```: 해당 값의 발생 빈도 계산
+ ```take(n)```: 처음 ```n```개의 값을 반환
+ ```first()```: 첫 번째 값을 반환
+ ```count()```: 값의 갯수를 계산

In [ ]:
"""
    [+] List로부터 RDD 객체 생성
    - SparkContext.parallelize()
    - 리스트 값: 
    "짜장면", "마라탕", "짬뽕", 
    "떡볶이", "쌀국수", "짬뽕", 
    "짜장면", "짜장면", "짜장면", 
    "라면", "우동", "라면"
"""

food_lst = [
    "짜장면", "마라탕", "짬뽕", 
    "떡볶이", "쌀국수", "짬뽕", 
    "짜장면", "짜장면", "짜장면", 
    "라면", "우동", "라면"]
       
# [+] lst로부터 RDD 객체 생성하기
foods = sc.parallelize(food_lst)

In [ ]:
# [+] foods RDD 내용 출력
foods.collect()

In [ ]:
# [+] RDD 원소 값 별로 발생 빈도 계산
foods.countByValue()

In [ ]:
# [+] take(n): 처음 n개의 값을 출력
food.take(n)

In [ ]:
# [+] first(): 첫 번째 값을 출력
food.first()

In [ ]:
# count(): 값 갯수 출력(중복된 원소 포함)
foods.count()

In [ ]:
# distinct(): unique한 원소로 구성된 RDD를 생성하는 Transformation

unique_foods = foods.distinct()
unique_foods.collect()

### **기본적인 변환 메서드들**

In [ ]:
# Transformations 은 지연 실행된다.
sc.parallelize([1, 2, 3]).map(lambda x: x + 2)

In [ ]:
# Actions 를 만나면 즉시 실행된다.
sc.parallelize([1, 2, 3]).map(lambda x: x + 2).collect()

In [ ]:
sc.parallelize([1, 2, 3]).map(lambda x: x * 2).collect()

In [ ]:
movies = [
    "그린 북",
    "매트릭스",
    "토이 스토리",
    "캐스트 어웨이",
    "포드 V 페라리",
    "보헤미안 랩소디",
    "빽 투 더 퓨처",
    "반지의 제왕",
    "죽은 시인의 사회"
]

In [ ]:
# [+] 리스트(movies) -> RDD 객체(moviesRDD) 생성
moviesRDD = sc.parallelize(movies)
moviesRDD.collect()

In [ ]:
# [+] flatMap을 이용한 Tokenizing
flatMovies = moviesRDD.flatMap(lambda movie: movie.split(" "))
flatMovies.collect()

In [ ]:
"""
    [+] flatMovies 에서 '매트릭스'를 제외
    - filter(lambda ...)
    - 저장할 객체: filteredMovies
"""

filteredMovies = flatMovies.filter(lambda x : x != '매트릭스')

In [ ]:
filteredMovies.collect()

### **집합 연산 관련 변환 메서드들**
+ ```intersection(rdd)```: 교집합 계산
+ ```union(rdd)```: 합집합 계산
+ ```subtract(rdd)```: 차집합 계산

In [ ]:
# [+] 두 개의 정수 리스트로부터 RDD 객체 생성(num1, num2)
num1 = sc.parallelize([1, 2, 3, 4, 5])
num2 = sc.parallelize([4, 5, 6, 7, 8])

In [ ]:
# [+] 교집합 구하기
num1.intersection(num2).collect()

In [ ]:
# [+] 합집합 구하기
num1.union(num2).collect()

In [ ]:
# [+] 차집합 구하기
num1.subtract(num2).collect()

### **랜덤 샘플링 함수**
+ ```sample(withReplacement, fraction)```: 원소를 랜덤 샘플링하여 새로운 RDD를 생성하는 narrow transformation
    + ```withReplacement```, bool[optional]: 중복 샘플링 허용 여부
    + ```fraction```, float[optional]: 샘플링 확률[0~1], 확률이므로 정확한 사이즈가 보장되지 않음
    + ```seed```: 결과 재현(reproducibility)을 위한 설정 값

In [ ]:
# [+] 합집합 생성(num1 + num2 -> numUnion)
numUnion = num1.union(num2)
numUnion.collect()

In [ ]:
# [+] 샘플링: 중복 허용, 50%
numUnion.sample(True, 0.5).collect()

In [ ]:
# [+] seed 파라미터를 설정하면 동일한 결과를 반복적으로 얻을 수 있다.
# [+] seed 파라미터를 설정하면 동일한 결과를 반복적으로 얻을 수 있다.
numUnion.sample(True, 0.5, parameter).collect()

### **그룹핑 연산**
+ ```groupBy()```: 특정 기준(함수로 정의)으로 그룹핑을 수행하는 wide transformation

In [ ]:
students = sc.parallelize([
    "이수경","이학선","이준솔",
    "이채원","윤지원","김태근",
    "서지유","이다영","정승윤",
    "박민규","이예소","이영림",
    "정승준","이용희","유형원",
    "박병규","방유리","박진영",
    "장민혁","김도훈"
])

In [ ]:
# [+] 원소의 첫 번째 값을 기준으로 그룹핑
studentGroup = students.groupBy(lambda x : x[0])
studentGroup.collect()

In [ ]:
# collect() 결과를 리스트 객체(res)에 저장
res = studentGroup.collect()
res

In [ ]:
res[0][1]

In [ ]:
# ResultIterable 객체는 list로 쉽게 변환 가능하다.
list(res[0][1])

In [ ]:
# [+] for 문을 이용하여 구조적으로 결과를 출력
for (k, v) in res:
    print(k, list(v))

In [ ]:
# groupBy()는 모드 연산과 같이 수학 연산에도 사용될 수 있다.
nums = sc.parallelize([1,2,3,4,5,6,7,8,9,10])

In [ ]:
# 홀수/짝수로 그룹핑
nums.groupBy(lambda x: x % 2).collect()

In [ ]:
res = nums.groupBy(lambda x: x % 2).collect()

In [ ]:
# [+] '1' 그룹에 해당되는 원소 리스트 출력하기
list(res[0][1])

In [ ]:
# [+] '0' 그룹에 해당되는 원소 리스트 출력하기
list(res[1][1])